# Matched low-light × weather diagnostic
Run with Internet + GPU T4. Save Version → Save & Run All, then the laptop can be closed. No training. The notebook downloads the full CDD-11 test ZIP and the same two frozen checkpoints, then evaluates only the 78 locked discovery scenes. Each clean scene shares one weather manifest across clean/low/weather/combined views. Two low-light strengths (0.6, 1.0), rain+haze and snow+haze. The measured remaining-error fraction is descriptive; it does not prove a novel restoration mechanism. Download `low_weather_counterfactual_results.zip` even if one model fails.


In [ ]:
import os, sys, subprocess, json, shutil, traceback
from pathlib import Path
REPO = Path('/kaggle/working/CoT-restoration')
URL = 'https://github.com/HoangKhanhTung0111/CoT-restoration.git'
if REPO.exists(): subprocess.run(['git','-C',str(REPO),'pull','--ff-only'],check=True)
else: subprocess.run(['git','clone','--depth','1',URL,str(REPO)],check=True)
os.chdir(REPO)
WORK = Path('/kaggle/working/low_weather_counterfactual')
WORK.mkdir(exist_ok=True)
print('Project revision:',subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip())
subprocess.run([sys.executable,'-m','pip','install','huggingface_hub','gdown','einops','timm','fvcore','thop','scikit-image'],check=True)
subprocess.run([sys.executable,'-m','unittest','hybrid_cot_nafnet.test_low_weather_interaction','-q'],check=True)
import torch
assert torch.cuda.is_available(), 'Enable a Kaggle GPU before running.'
print(torch.__version__,torch.cuda.get_device_name(0))


In [ ]:
import os, sys, subprocess, json, shutil, traceback
from pathlib import Path
REPO = Path('/kaggle/working/CoT-restoration')
WORK = Path('/kaggle/working/low_weather_counterfactual')
BUNDLE = WORK/'bundle'
BUNDLE.mkdir(parents=True,exist_ok=True)
errors = []
def run_logged(args,name):
    with (BUNDLE/(name+'.log')).open('w') as log:
        process=subprocess.Popen([sys.executable,'-u','-m',*args,'--work',str(WORK)],cwd=REPO,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in process.stdout: print(line,end=''); log.write(line); log.flush()
        if process.wait()!=0: raise RuntimeError(name+' failed; send result ZIP')
try:
    run_logged(['hybrid_cot_nafnet.common_failure_audit','prepare'],'prepare')
    for model in ['onerestore','mirage']:
        try: run_logged(['hybrid_cot_nafnet.audit_low_weather_interaction','--model',model],model)
        except Exception as exc: errors.append(str(exc))
except Exception: errors.append(traceback.format_exc())
finally:
    for name in ['manifest.json','sources.json']:
        if (WORK/name).exists(): shutil.copy2(WORK/name,BUNDLE/name)
    if (WORK/'interaction').exists(): shutil.copytree(WORK/'interaction',BUNDLE/'interaction',dirs_exist_ok=True)
    (BUNDLE/'run.json').write_text(json.dumps({'errors':errors,'project_commit':subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()},indent=2))
    (BUNDLE/'environment.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True))
    archive=shutil.make_archive('/kaggle/working/low_weather_counterfactual_results','zip',BUNDLE)
    print('Download:',archive)
if errors: raise RuntimeError('Diagnostic incomplete; send the ZIP: '+str(errors))
print('Matched diagnostic complete. Interpretation requires scene-level analysis.')
